# Government Scheme Summarizer

Summarize Indian government welfare schemes into plain-language bullet points, then translate the summary into an Indian language, using Sarvam's Chat Completions and Translate APIs.

Pipeline overview:
1. Ask `sarvam-105b` (with wiki grounding) to summarize a scheme name or description
2. Chunk and translate the summary into a target Indian language with the Translate API
3. Save the English and translated summaries to `outputs/`

In [ ]:
%pip install -r requirements.txt

## Setup

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

from dotenv import load_dotenv
from sarvamai import SarvamAI

load_dotenv()

SARVAM_API_KEY = os.getenv("SARVAM_API_KEY")
if not SARVAM_API_KEY:
    raise RuntimeError(
        "Set SARVAM_API_KEY in your environment or .env file before running."
    )

client = SarvamAI(api_subscription_key=SARVAM_API_KEY)
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)


## Supported languages

Target languages for the Translate API (`mayura:v1`). Codes follow BCP-47 with the `-IN` suffix.

In [ ]:
SUPPORTED_LANGUAGES = {
    "hi-IN": "Hindi",
    "ta-IN": "Tamil",
    "te-IN": "Telugu",
    "bn-IN": "Bengali",
    "ml-IN": "Malayalam",
    "kn-IN": "Kannada",
    "mr-IN": "Marathi",
    "gu-IN": "Gujarati",
    "pa-IN": "Punjabi",
    "od-IN": "Odia",
}


## Chunk long text for translation

`mayura:v1` accepts a limited number of characters per request, so long summaries are split on word boundaries before translation.

In [ ]:
def chunk_text(text: str, max_length: int = 2000) -> list[str]:
    """Split text into chunks of at most max_length characters, preserving word boundaries."""
    chunks = []
    remaining = text

    while len(remaining) > max_length:
        split_at = remaining.rfind(" ", 0, max_length)
        if split_at == -1:
            split_at = max_length
        chunks.append(remaining[:split_at].strip())
        remaining = remaining[split_at:].strip()

    if remaining:
        chunks.append(remaining)
    return chunks


## Summarize a scheme with the Chat Completions API

`wiki_grounding=True` lets the model ground its answer in up-to-date public information rather than relying solely on parametric knowledge.

In [ ]:
def summarize_scheme(scheme: str, category: str = "") -> str:
    """Summarize a government scheme in bullet points using sarvam-105b."""
    system_prompt = (
        "You are an Indian government official who is an expert in this scheme "
        "category and helps Indian citizens understand Indian government schemes."
    )
    if category:
        system_prompt += f" Category: {category}."

    user_prompt = (
        f"Summarize the scheme '{scheme}' in bullet points in no more than 1500 characters, "
        "and include official links or reliable government sources for further information."
    )

    response = client.chat.completions(
        model="sarvam-105b",
        max_tokens=2000,
        wiki_grounding=True,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )
    return response.choices[0].message.content


## Translate the summary with the Translate API

Each chunk is translated independently and the results are joined back together.

In [ ]:
def translate_summary(summary: str, target_language_code: str) -> str:
    """Translate a scheme summary into an Indian language, chunk by chunk."""
    translated_chunks = []
    for chunk in chunk_text(summary):
        response = client.text.translate(
            input=chunk,
            source_language_code="en-IN",
            target_language_code=target_language_code,
            speaker_gender="Male",
            mode="classic-colloquial",
            model="mayura:v1",
        )
        translated_chunks.append(response.translated_text)
    return "\n".join(translated_chunks)


## Run the pipeline

Summarize a scheme, translate it into Hindi, and save both versions to `outputs/`.

In [ ]:
SCHEME_NAME = "Pradhan Mantri Jan Dhan Yojana"
CATEGORY = "Banking, Financial Services & Insurance (BFSI)"
TARGET_LANGUAGE = "hi-IN"

summary = summarize_scheme(SCHEME_NAME, CATEGORY)
print(f"English summary for '{SCHEME_NAME}':\n{summary}\n")

translated_summary = translate_summary(summary, TARGET_LANGUAGE)
print(f"{SUPPORTED_LANGUAGES[TARGET_LANGUAGE]} summary:\n{translated_summary}\n")

result = {
    "scheme": SCHEME_NAME,
    "category": CATEGORY,
    "summary_en": summary,
    "target_language": TARGET_LANGUAGE,
    "summary_translated": translated_summary,
}

output_path = OUTPUT_DIR / f"{SCHEME_NAME.replace(' ', '_').lower()}.json"
output_path.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Saved result to {output_path}")


## Next steps

- Try a different `SCHEME_NAME`, `CATEGORY`, or `TARGET_LANGUAGE` (see `SUPPORTED_LANGUAGES` above)
- Loop over multiple languages to generate summaries for every citizen-facing language you support
- Read more in the [Chat Completions docs](https://docs.sarvam.ai/api-reference-docs/chat/completions) and [Translate docs](https://docs.sarvam.ai/api-reference-docs/text/translate)